## Binomial Option Pricing

This notebook implements one-period and multi-period binomial option pricing using replicating portfolios and risk-neutral valuation. The binomial model provides a discrete-time foundation for the continuous-time Black–Scholes framework.

We define:

- $S_0$: initial stock price
- u: up multiplier
- d: down multiplier
- K: strike price
- r: gross risk-free return per period (rather than continuously compounded interest rate used in Black-Scholes)
- $C_u$: call payoff in up state
- $C_d$: call payoff in down state

The no-arbitrage condition is $0<d<r<u$

In [ ]:
S0 = 100
u = 1.2
d = 0.8
K = 100
r = 1.05
n = 3

### One-Period Binomial Model

At maturity $T$, European call and put payoffs are $\, C_T=\max(S_T-K,0), \, P_T=\max(K-S_T,0)$

After one period, the stock price is either $uS_0$ or $dS_0$, giving call payoffs $\, C_u=\max(uS_0-K,0), \, C_d=\max(dS_0-K,0)$

In [8]:
Cu = max(u * S0 - K, 0)
Cd = max(d * S0 - K, 0)


### Replicating Portfolio

Consider a portfolio holding $\Delta$ shares and $B$ in the risk-free asset. Its initial value is

$$
V_0=\Delta S_0+B.
$$

To replicate the option payoff in both states,

$$
\begin{aligned}
\Delta uS_0+rB &= C_u,\\
\Delta dS_0+rB &= C_d.
\end{aligned}
$$

Solving simultaneously,

$$
\Delta^*=\frac{C_u-C_d}{S_0(u-d)},\qquad
B^*=\frac{uC_d-dC_u}{r(u-d)}.
$$

Under no arbitrage, the option price equals the initial cost of the replicating portfolio.


In [9]:
delta_star = (Cu - Cd)/(S0*(u-d))
B_star = (u*Cd - d*Cu)/r*(u-d)

In [ ]:
C0 = delta_star * S0 + B_star


### Risk-Neutral Valuation

Substituting the replicating portfolio holdings gives

$$
C_0=\Delta^*S_0+B^*
=\frac{1}{r}\left(qC_u+(1-q)C_d\right),
$$

where

$$
q=\frac{r-d}{u-d}.
$$

Here $q$ is the risk-neutral probability of an up move, rather than the actual probability. The option price is the discounted expected payoff under this probability.


It must follow, through the no-arbitrage condition, that

$$
C_0 = \Delta^* S_0 + B^* =
\frac{1}{r}
\left[
\frac{r-d}{u-d}C_u
+
\frac{u-r}{u-d}C_d
\right]
$$

### Multi-Period Binomial Model

Extending the model to $n$ periods, a path containing $k$ up moves has terminal stock price

$$
S_T=S_0u^kd^{n-k}.
$$

The European call price is

$$
C_0=\frac{1}{r^n}\sum_{k=0}^{n}
\binom{n}{k}q^k(1-q)^{n-k}
\max(S_0u^kd^{n-k}-K,0).
$$

Implementation below uses backward induction, starting from terminal payoffs and recursively applying

$$
C=\frac{qC_u+(1-q)C_d}{r}
$$

until the time-zero option value is obtained.


In [ ]:
def binomial_price(S0, K, u, d, r, n):

    q = (r - d)/(u - d)

    Sn_list = []

    for k in range(n+1):
        Sn = u**k * d**(n-k) * S0
        Sn_list.append(Sn)

    Cn_list = []

    for i in range(len(Sn_list)):
        Cn = max(Sn_list[i] - K, 0)
        Cn_list.append(Cn)

    current_values = Cn_list

    for k in range(1,n + 1):

        previous_values = []
        
        for i in range(len(Cn_list) - k):
            x = 1/r * (q * (current_values[i+1]) + (1-q) * (current_values[i]))
            previous_values.append(x)

        current_values = previous_values

    return current_values[0]


### Conclusion

The binomial model prices European options through replication and risk-neutral valuation. Backward induction extends the one-period argument to multiple periods.

Brownian motion and geometric Brownian motion provide the continuous-time framework used in the subsequent Black–Scholes model.